Named Entity Recognition (NER) System

In [1]:
# Import Required Libraries
import spacy
from collections import defaultdict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Load spaCy English model (pre-trained NER model)
try:
    nlp = spacy.load("en_core_web_sm")
    print("✓ spaCy model loaded successfully")
except OSError:
    print("Downloading spaCy English model...")
    import os
    os.system("python -m spacy download en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")
    print("✓ spaCy model downloaded and loaded")

✓ spaCy model downloaded and loaded


In [2]:
# Sample Dataset: News Articles and Social Media Text
sample_texts = [
    "Apple Inc. is planning to open a new office in San Francisco next year.",
    "Elon Musk announced Tesla's electric vehicle launch in Germany.",
    "Microsoft CEO Satya Nadella visited India for a tech conference.",
    "Google announced new AI features at their annual developer conference.",
    "Meta founder Mark Zuckerberg discussed the metaverse strategy on Twitter.",
    "Amazon is expanding its AWS services to the Middle East.",
    "Netflix is investing heavily in Indian content production.",
    "OpenAI released GPT-4 on Tuesday, revolutionizing AI.",
    "IBM partnered with Nokia for 5G technology development.",
    "Oracle announced a new cloud computing platform."
]

# Ground truth annotations (for evaluation)
# Format: {text_index: [(entity_text, label), ...]}
ground_truth = {
    0: [("Apple Inc.", "ORG"), ("San Francisco", "GPE")],
    1: [("Elon Musk", "PERSON"), ("Tesla", "ORG"), ("Germany", "GPE")],
    2: [("Microsoft", "ORG"), ("Satya Nadella", "PERSON"), ("India", "GPE")],
    3: [("Google", "ORG")],
    4: [("Meta", "ORG"), ("Mark Zuckerberg", "PERSON"), ("Twitter", "ORG")],
    5: [("Amazon", "ORG"), ("Middle East", "GPE")],
    6: [("Netflix", "ORG"), ("Indian", "NORP")],
    7: [("OpenAI", "ORG"), ("GPT-4", "PRODUCT")],
    8: [("IBM", "ORG"), ("Nokia", "ORG")],
    9: [("Oracle", "ORG")]
}

print(f"✓ Loaded {len(sample_texts)} sample texts for NER")
print("\nSample texts:")
for i, text in enumerate(sample_texts[:3]):
    print(f"  {i+1}. {text}")

✓ Loaded 10 sample texts for NER

Sample texts:
  1. Apple Inc. is planning to open a new office in San Francisco next year.
  2. Elon Musk announced Tesla's electric vehicle launch in Germany.
  3. Microsoft CEO Satya Nadella visited India for a tech conference.


In [3]:
# Extract Named Entities using spaCy NER
predicted_entities = {}

print("\n" + "="*70)
print("NAMED ENTITY RECOGNITION RESULTS")
print("="*70)

for i, text in enumerate(sample_texts):
    doc = nlp(text)
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    predicted_entities[i] = entities
    
    print(f"\nText {i+1}: {text}")
    print(f"Predicted Entities: {entities if entities else 'None'}")
    print(f"Ground Truth:      {ground_truth[i]}")

print("\n" + "="*70)


NAMED ENTITY RECOGNITION RESULTS

Text 1: Apple Inc. is planning to open a new office in San Francisco next year.
Predicted Entities: [('Apple Inc.', 'ORG'), ('San Francisco', 'GPE'), ('next year', 'DATE')]
Ground Truth:      [('Apple Inc.', 'ORG'), ('San Francisco', 'GPE')]

Text 2: Elon Musk announced Tesla's electric vehicle launch in Germany.
Predicted Entities: [('Elon Musk', 'PERSON'), ('Tesla', 'ORG'), ('Germany', 'GPE')]
Ground Truth:      [('Elon Musk', 'PERSON'), ('Tesla', 'ORG'), ('Germany', 'GPE')]

Text 3: Microsoft CEO Satya Nadella visited India for a tech conference.
Predicted Entities: [('Microsoft', 'ORG'), ('Satya Nadella', 'PERSON'), ('India', 'GPE')]
Ground Truth:      [('Microsoft', 'ORG'), ('Satya Nadella', 'PERSON'), ('India', 'GPE')]

Text 4: Google announced new AI features at their annual developer conference.
Predicted Entities: [('Google', 'ORG'), ('AI', 'GPE'), ('annual', 'DATE')]
Ground Truth:      [('Google', 'ORG')]

Text 5: Meta founder Mark Zuckerber

In [4]:
# Evaluation Metrics Calculation
def evaluate_ner(predicted, ground_truth):
    """
    Calculate precision, recall, and F1 score for NER predictions
    """
    total_predicted = 0
    total_correct = 0
    total_ground_truth = 0
    
    # Convert to sets for comparison
    for idx in predicted:
        pred_set = set(predicted[idx])
        truth_set = set(ground_truth[idx])
        
        total_predicted += len(pred_set)
        total_ground_truth += len(truth_set)
        total_correct += len(pred_set & truth_set)
    
    # Calculate metrics
    precision = total_correct / total_predicted if total_predicted > 0 else 0
    recall = total_correct / total_ground_truth if total_ground_truth > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = total_correct / max(total_ground_truth, total_predicted) if max(total_ground_truth, total_predicted) > 0 else 0
    
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'accuracy': accuracy,
        'correct': total_correct,
        'predicted': total_predicted,
        'ground_truth': total_ground_truth
    }

# Evaluate the model
metrics = evaluate_ner(predicted_entities, ground_truth)

print("\n" + "="*70)
print("PERFORMANCE METRICS")
print("="*70)
print(f"\nTotal Predicted Entities:    {metrics['predicted']}")
print(f"Total Ground Truth Entities: {metrics['ground_truth']}")
print(f"Correctly Identified:        {metrics['correct']}")
print(f"\n{'Metric':<15} {'Score':<10}")
print("-" * 25)
print(f"{'Accuracy':<15} {metrics['accuracy']:.4f}")
print(f"{'Precision':<15} {metrics['precision']:.4f}")
print(f"{'Recall':<15} {metrics['recall']:.4f}")
print(f"{'F1 Score':<15} {metrics['f1']:.4f}")
print("="*70)


PERFORMANCE METRICS

Total Predicted Entities:    27
Total Ground Truth Entities: 21
Correctly Identified:        16

Metric          Score     
-------------------------
Accuracy        0.5926
Precision       0.5926
Recall          0.7619
F1 Score        0.6667


In [5]:
# Entity Type-wise Analysis
entity_stats = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})

for idx in predicted_entities:
    pred_dict = {text: label for text, label in predicted_entities[idx]}
    truth_dict = {text: label for text, label in ground_truth[idx]}
    
    # True Positives and False Negatives
    for text, label in ground_truth[idx]:
        if text in pred_dict and pred_dict[text] == label:
            entity_stats[label]['tp'] += 1
        else:
            entity_stats[label]['fn'] += 1
    
    # False Positives
    for text, label in predicted_entities[idx]:
        if text not in truth_dict or truth_dict[text] != label:
            entity_stats[label]['fp'] += 1

print("\n" + "="*70)
print("ENTITY TYPE-WISE PERFORMANCE")
print("="*70)
print(f"\n{'Entity Type':<15} {'TP':<6} {'FP':<6} {'FN':<6} {'Precision':<12} {'Recall':<12}")
print("-" * 65)

for entity_type, stats in sorted(entity_stats.items()):
    tp, fp, fn = stats['tp'], stats['fp'], stats['fn']
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    print(f"{entity_type:<15} {tp:<6} {fp:<6} {fn:<6} {precision:<12.4f} {recall:<12.4f}")

print("="*70)
print("\nLegend:")
print("  TP (True Positives):  Correctly identified entities")
print("  FP (False Positives): Incorrectly identified as entities")
print("  FN (False Negatives): Actual entities that were missed")


ENTITY TYPE-WISE PERFORMANCE

Entity Type     TP     FP     FN     Precision    Recall      
-----------------------------------------------------------------
CARDINAL        0      1      0      0.0000       0.0000      
DATE            0      3      0      0.0000       0.0000      
GPE             3      3      1      0.5000       0.7500      
LOC             0      1      0      0.0000       0.0000      
NORP            1      0      0      1.0000       1.0000      
ORG             9      1      3      0.9000       0.7500      
PERSON          3      1      0      0.7500       1.0000      
PRODUCT         0      1      1      0.0000       0.0000      

Legend:
  TP (True Positives):  Correctly identified entities
  FP (False Positives): Incorrectly identified as entities
  FN (False Negatives): Actual entities that were missed
